# Options Market Stream PoC — `nbstream.binance.com`

BTC, ETH 등 가상자산을 기초자산으로 하는 유러피언 옵션이다.

```text
nbstream.binance.com/eoptions
 ├─ C - Call Option   예: BTC-260925-100000-C
 └─ P - Put Option    예: BTC-260925-100000-P
```

심볼 형식은 `기초자산-만기(YYMMDD)-행사가-타입`이다. **만기와 행사가 조합이 계속 새로 생기고
사라지므로 심볼을 하드코딩하면 곧 존재하지 않는 심볼이 된다.** 이 노트북은 `exchangeInfo`로
현재 거래 가능한 심볼을 조회하는 것부터 시작한다.

> **참고 (2026-08-26 확인):** REST 엔드포인트(`/eapi/v1/*`)는 정상 동작하지만, 이 환경에서는
> 공식 문서의 WebSocket 스트림(`wss://nbstream.binance.com/eoptions/ws/...`)이 handshake 단계에서
> AWS ELB로부터 `404`를 받았다. 옵션 상품은 지역별 접근 제한이 있을 수 있어 네트워크/리전에 따른
> 차단으로 추정된다. 그래서 이 노트북은 **REST 스냅샷을 우선 확인**하고, WebSocket은 실패해도
> 노트북이 죽지 않도록 예외 처리를 해서 별도로 시도한다.

- 관련 문서: [docs/binanceWebSocketREADME.md](../../binance-api/binanceWebSocketREADME.md)


In [ ]:
%pip install -q "websockets>=17,<18"

In [ ]:
import asyncio
import json
import urllib.request
from datetime import datetime, timezone

from websockets.asyncio.client import connect

WS_BASE = "wss://nbstream.binance.com/eoptions/ws"
REST_BASE = "https://eapi.binance.com"
UNDERLYING = "BTCUSDT"


def get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=10) as resp:
        return json.load(resp)

## 1. 현재 거래 가능한 옵션 심볼 조회

- REST: `GET /eapi/v1/exchangeInfo`
- 응답의 `optionSymbols` 배열에서 `underlying`이 `BTCUSDT`이고 `status`가 `TRADING`인
  항목만 남긴다.


In [ ]:
exchange_info = get_json(f"{REST_BASE}/eapi/v1/exchangeInfo")
btc_options = [
    s for s in exchange_info["optionSymbols"]
    if s.get("underlying") == UNDERLYING and s.get("status") == "TRADING"
]
len(btc_options)

## 2. 만기가 가장 가까운 Call / Put 고르기

`expiryDate`(ms epoch) 기준으로 정렬해서 가장 가까운 만기의 Call 1개, Put 1개를 고른다.


In [ ]:
def nearest(side: str) -> dict:
    candidates = sorted(
        (s for s in btc_options if s["side"] == side),
        key=lambda s: (s["expiryDate"], float(s["strikePrice"])),
    )
    return candidates[0]


call_option = nearest("CALL")
put_option = nearest("PUT")


def describe(option: dict) -> str:
    expiry = datetime.fromtimestamp(option["expiryDate"] / 1000, tz=timezone.utc)
    return f"{option['symbol']}  strike={option['strikePrice']}  expiry={expiry:%Y-%m-%d}"


print(describe(call_option))
print(describe(put_option))

## 3. REST Ticker로 시세 확인

- REST: `GET /eapi/v1/ticker?symbol=<symbol>`
- 주요 필드: `lastPrice`, `bidPrice`, `askPrice`, `strikePrice`, `exercisePrice`(기초자산 현재가)


In [ ]:
call_ticker = get_json(f"{REST_BASE}/eapi/v1/ticker?symbol={call_option['symbol']}")
call_ticker

## 4. WebSocket 스트림 구독 시도

공식 문서 기준 URL 형식이다. 연결이 안 되면(예: 404) 예외를 잡아서 REST로 대체했다는 메시지만
출력하고 빈 리스트를 반환한다 — 이 코드 자체가 틀린 것이 아니라 네트워크/리전 문제일 수 있다.

- 스트림: `<symbol>@ticker`, `<symbol>@trade`


In [ ]:
async def receive(stream: str, message_count: int = 3, timeout: float = 15) -> list[dict]:
    url = f"{WS_BASE}/{stream}"
    messages = []

    try:
        async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
            for _ in range(message_count):
                try:
                    message = await asyncio.wait_for(ws.recv(), timeout=timeout)
                except TimeoutError:
                    print(f"{timeout}초 동안 추가 메시지 없음: {len(messages)}건 수신")
                    break
                messages.append(json.loads(message))
    except Exception as exc:
        print(f"WebSocket 연결 실패({exc!r}) — REST 스냅샷으로 대체 확인한다.")

    return messages


option_ticker_messages = await receive(f"{call_option['symbol']}@ticker", message_count=3)
option_ticker_messages

## 5. Underlying Index Stream

옵션 상품 전용으로, 기초자산(BTC)의 지수 가격을 전달한다. 특정 옵션 심볼의 유동성과 무관하게
갱신된다.

- 스트림: `<underlying>@index`


In [ ]:
index_messages = await receive(f"{UNDERLYING}@index", message_count=3)
index_messages